<a href="https://colab.research.google.com/github/jef4j/Notebook_Actividad_S01/blob/main/Jefferson_Arrieta_Notebook_Actividad_S02_Limpieza_Datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Diplomado de Grado en Análisis de Datos Aplicado a la Toma de Decisiones
## UNICOLOMBO — Fundación Universitaria Colombo Internacional · Cartagena
**Módulo 2:** Preparación y Análisis Exploratorio de Datos  
**Sesión 2:** Limpieza de Datos  
**Docente:** Ing. Heyder Medrano Olier | **Período:** 2026 — Vacaciones

---
**Tipo:** Notebook de Actividad  
**Entrega:** 28 de junio de 2026 · 23:59 (hora Colombia)  
**Valor:** 5.0 puntos  

**Estudiante:** [Escriba su nombre completo]  

> Aplique el proceso completo de limpieza al dataset DataRetail LATAM.
> Cada decisión debe estar justificada en celdas Markdown.
> Al finalizar: *Kernel → Restart & Run All* y verificar que no haya errores.

## 1. Librerías

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import missingno as msno, warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
print('OK')

## 2. Cargar Dataset

In [ ]:
import numpy as np, pandas as pd, random
from datetime import datetime, timedelta
np.random.seed(42); random.seed(42)
N=2000
ciudades=["Bogotá","Medellín","Cali","Barranquilla","Cartagena",
          "Lima","Ciudad de México","Buenos Aires","Santiago","Montevideo"]
ciu_s=ciudades+["bogota","BOGOTÁ","Medellin","cali ","barranquilla","SANTIAGO"]
paises=["Colombia","Perú","México","Argentina","Chile","Uruguay"]
cats=["Computación","Periféricos","Audio","Telefonía","Accesorios","Gaming","Componentes"]
canales=["Tienda Física","E-commerce","Distribuidor","Corporativo","Marketplace"]
can_s=canales+["tienda fisica","E-Comerce","distribuidor ","CORPORATIVO"]
prods=["Laptop Pro","Tablet Ultra","Monitor 4K","Teclado Inalámbrico",
       "Auriculares BT","Parlante Portátil","Smartphone Galaxy",
       "Cámara Web HD","Smartwatch Pro","Mouse Gaming","SSD 1TB","GPU RTX 4060"]
precios={"Laptop Pro":1200,"Tablet Ultra":450,"Monitor 4K":380,"Teclado Inalámbrico":75,
         "Auriculares BT":120,"Parlante Portátil":90,"Smartphone Galaxy":680,
         "Cámara Web HD":95,"Smartwatch Pro":250,"Mouse Gaming":65,"SSD 1TB":130,"GPU RTX 4060":580}
rows=[]
for i in range(N):
    prod=random.choice(prods); cat=cats[prods.index(prod)%len(cats)]
    pais=random.choice(paises)
    ciudad=random.choice(ciu_s) if random.random()<0.15 else random.choice(ciudades)
    canal=random.choice(can_s) if random.random()<0.10 else random.choice(canales)
    qty=random.randint(1,20); precio=round(precios[prod]*np.random.uniform(0.8,1.3),2)
    desc=round(random.choice([0,0,0,0.05,0.10,0.15,0.20,0.30,0.35,0.50]),2)
    total=round(qty*precio*(1-desc),2); margen=round(total*np.random.uniform(-0.05,0.35),2)
    fecha=datetime(2024,1,1)+timedelta(days=random.randint(0,729))
    rows.append({"id_venta":f"V{i+1:05d}","fecha_venta":fecha.strftime("%Y-%m-%d"),
                 "id_cliente":f"C{random.randint(1,500):04d}",
                 "nombre_cliente":f"Cliente {random.randint(1,500)}",
                 "ciudad":ciudad,"pais":pais,"id_producto":f"P{prods.index(prod)+1:02d}",
                 "nombre_producto":prod,"categoria":cat,"canal_venta":canal,
                 "cantidad":qty,"precio_unitario":precio,"descuento":desc,
                 "total_venta":total,"margen_utilidad":margen})
df_raw=pd.DataFrame(rows)
idx_p=np.random.choice(df_raw.index,size=int(N*0.06),replace=False)
idx_t=np.random.choice(df_raw.index,size=int(N*0.04),replace=False)
idx_m=np.random.choice(df_raw.index,size=int(N*0.03),replace=False)
df_raw.loc[idx_p,"precio_unitario"]=np.nan
df_raw.loc[idx_t,"total_venta"]=np.nan
df_raw.loc[idx_m,"margen_utilidad"]=np.nan
dup_idx=np.random.choice(df_raw.index,size=40,replace=False)
df_raw=pd.concat([df_raw,df_raw.loc[dup_idx].copy()],ignore_index=True)
out_idx=np.random.choice(df_raw.index,size=10,replace=False)
df_raw.loc[out_idx,"total_venta"]=np.random.uniform(50000,200000,10)
df_raw=df_raw.sample(frac=1,random_state=42).reset_index(drop=True)
df=df_raw.copy()
df["fecha_venta"]=pd.to_datetime(df["fecha_venta"],errors="coerce")
df["precio_unitario"]=pd.to_numeric(df["precio_unitario"],errors="coerce")
print(f"Dataset: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"Nulos: {df.isna().sum().sum():,} | Duplicados: {df.duplicated().sum():,}")
df.head(3)


## 3. Registrar Estado Inicial

Antes de cualquier modificación, registre los conteos de problemas detectados en S01.

In [ ]:
ri = {'filas':len(df),'nulos_precio':df['precio_unitario'].isna().sum(),
      'nulos_total':df['total_venta'].isna().sum(),'nulos_margen':df['margen_utilidad'].isna().sum(),
      'duplicados':df.duplicated().sum(),'desc_viol':(df['descuento']>0.30).sum()}
print('ESTADO INICIAL (antes de limpiar):')
for k,v in ri.items(): print(f'  {k:<20}: {v:,}')

## 4. Corrección de Tipos

> Justifique por qué se deben corregir los tipos ANTES de cualquier otra operación.

In [ ]:
df['cantidad']  = pd.to_numeric(df['cantidad'],  errors='coerce').astype('Int64')
df['descuento'] = pd.to_numeric(df['descuento'], errors='coerce')
print('Tipos corregidos:', df.dtypes[['cantidad','precio_unitario','descuento','total_venta']].to_dict())

### 📝 Justificación
> [Su respuesta aquí — ¿qué problema específico resuelve la corrección de tipos?]

## 5. Eliminación de Duplicados

In [ ]:
n0 = len(df)
df = df.drop_duplicates(keep='first').reset_index(drop=True)
print(f'Eliminados: {n0-len(df):,} | Quedan: {len(df):,}')

### 📝 Justificación del `keep='first'`
> [¿Por qué esta estrategia? ¿Cuándo usaría `keep=False`?]

## 6. Imputación de Nulos

Aplique las tres estrategias vistas en clase y justifique cada una.

### 6.1 precio_unitario — Mediana por Categoría

In [ ]:
n0 = df['precio_unitario'].isna().sum()
med_cat = df.groupby('categoria')['precio_unitario'].transform('median')
df['precio_unitario'] = df['precio_unitario'].fillna(med_cat)
df['precio_unitario'] = df['precio_unitario'].fillna(df['precio_unitario'].median())
print(f'Nulos precio: {n0} → {df["precio_unitario"].isna().sum()}')

### 6.2 total_venta — Recálculo

In [ ]:
n0 = df['total_venta'].isna().sum()
mask = df['total_venta'].isna()
df.loc[mask,'total_venta'] = (
    df.loc[mask,'cantidad'] * df.loc[mask,'precio_unitario'] * (1-df.loc[mask,'descuento'])
).round(2)
print(f'Nulos total: {n0} → {df["total_venta"].isna().sum()}')

### 6.3 margen_utilidad — Mediana por Categoría

In [ ]:
n0 = df['margen_utilidad'].isna().sum()
med_m = df.groupby('categoria')['margen_utilidad'].transform('median')
df['margen_utilidad'] = df['margen_utilidad'].fillna(med_m)
df['margen_utilidad'] = df['margen_utilidad'].fillna(df['margen_utilidad'].median())
print(f'Nulos margen: {n0} → {df["margen_utilidad"].isna().sum()}')
print(f'Nulos totales: {df.isna().sum().sum():,}')

## 7. Visualización Missingno — Antes vs Después

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,4))
fig.suptitle('Nulos: Antes vs Después', fontweight='bold')
df_v = df_raw.copy()
df_v['precio_unitario']=pd.to_numeric(df_v['precio_unitario'],errors='coerce')
msno.bar(df_v, ax=axes[0], color='#e15759', fontsize=9)
axes[0].set_title('ANTES')
msno.bar(df, ax=axes[1], color='#59a14f', fontsize=9)
axes[1].set_title('DESPUÉS')
plt.tight_layout(); plt.show()

## 8. Tratamiento de Outliers — Capping IQR

In [ ]:
def cap_iqr(serie, nombre):
    Q1,Q3 = serie.quantile(.25), serie.quantile(.75)
    IQR=Q3-Q1; n=((serie<Q1-1.5*IQR)|(serie>Q3+1.5*IQR)).sum()
    print(f'{nombre}: {n} outliers capeados')
    return serie.clip(Q1-1.5*IQR, Q3+1.5*IQR)

df['total_venta']     = cap_iqr(df['total_venta'],     'total_venta')
df['precio_unitario'] = cap_iqr(df['precio_unitario'], 'precio_unitario')

### 📝 Capping vs Eliminación
> [¿Cuándo es preferible capear y cuándo eliminar el outlier? Dé un ejemplo con estos datos.]

## 9. Normalización Categórica

In [ ]:
df['ciudad'] = df['ciudad'].str.strip().str.title()
mapa = {'Bogota':'Bogotá','Ciudad De Mexico':'Ciudad de México'}
df['ciudad'] = df['ciudad'].replace(mapa)
df['canal_venta'] = df['canal_venta'].str.strip()
mapa_c = {'tienda fisica':'Tienda Física','E-Comerce':'E-commerce',
          'distribuidor ':'Distribuidor','CORPORATIVO':'Corporativo'}
df['canal_venta'] = df['canal_venta'].replace(mapa_c)
print(f'Ciudades únicas: {df["ciudad"].nunique()} | Canales únicos: {df["canal_venta"].nunique()}')

## 10. Corrección de Dominio

In [ ]:
n_desc = (df['descuento']>0.30).sum()
df['descuento'] = df['descuento'].clip(upper=0.30)
df['total_venta'] = (df['cantidad']*df['precio_unitario']*(1-df['descuento'])).round(2)
print(f'Descuentos > 30% corregidos: {n_desc}')

## 11. Panel Visual

In [ ]:
fig, axes = plt.subplots(2,2,figsize=(14,9))
fig.suptitle('Panel Limpieza S02 — Antes vs Después', fontweight='bold')
pd.to_numeric(df_raw['total_venta'],errors='coerce').dropna().hist(ax=axes[0,0],bins=50,color='#e15759',ec='white')
axes[0,0].set_title('total_venta ANTES')
df['total_venta'].hist(ax=axes[0,1],bins=50,color='#59a14f',ec='white')
axes[0,1].set_title('total_venta DESPUÉS')
df_raw['ciudad'].value_counts().head(8).plot(kind='bar',ax=axes[1,0],color='#e15759',ec='white')
axes[1,0].set_title('ciudad ANTES'); axes[1,0].tick_params(axis='x',rotation=45)
df['ciudad'].value_counts().head(8).plot(kind='bar',ax=axes[1,1],color='#59a14f',ec='white')
axes[1,1].set_title('ciudad DESPUÉS'); axes[1,1].tick_params(axis='x',rotation=45)
plt.tight_layout(); plt.show()

## 12. Comparación Final y Score

In [ ]:
rf = {'filas':len(df),'nulos_precio':df['precio_unitario'].isna().sum(),
      'nulos_total':df['total_venta'].isna().sum(),'nulos_margen':df['margen_utilidad'].isna().sum(),
      'duplicados':df.duplicated().sum(),'desc_viol':(df['descuento']>0.30).sum()}
print(f'{"Métrica":<20} {"Antes":>8} {"Después":>9}')
print('-'*42)
for k in rf:
    print(f'{k:<20} {ri.get(k,0):>8,} {rf[k]:>9,}')

s = df['total_venta']
Q1,Q3=s.quantile(.25),s.quantile(.75)
po=((s<Q1-1.5*(Q3-Q1))|(s>Q3+1.5*(Q3-Q1))).mean()*100
score=100-min(df.isna().mean().mean()*300,30)-min(po*1.5,15)
print(f'Score: {score:.1f}/100')

## 13. Exportar Dataset Limpio

In [ ]:
df.to_csv('DataRetail_LATAM_S02_limpio.csv',index=False,encoding='utf-8-sig')
print(f'Exportado: {len(df):,} filas x {df.shape[1]} columnas')

## 14. Reflexiones y Conclusiones

### Mis 5 hallazgos más importantes:

1. [Su hallazgo 1]
2. [Su hallazgo 2]
3. [Su hallazgo 3]
4. [Su hallazgo 4]
5. [Su hallazgo 5]

### Impacto de negocio:
> [¿Qué problemas de negocio resuelve este dataset limpio para DataRetail LATAM?]

## Referencias
1. McKinney, W. (2022). *Python for Data Analysis* (3.ª ed.). O'Reilly.
2. Little, R. & Rubin, D. (2019). *Statistical Analysis with Missing Data* (3.ª ed.). Wiley.
3. García, S. et al. (2015). *Data Preprocessing in Data Mining*. Springer.
4. DAMA International (2017). *DAMA-DMBOK* (2.ª ed.).
5. Bilogur, A. (2023). missingno. https://github.com/ResidentMario/missingno

---
*Notebook Actividad S02 — UNICOLOMBO · 2026*